LeadIQ lead scoring model V1

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd, numpy as np
session = get_active_session()

df = session.sql("""
    SELECT t.* EXCLUDE (LABEL), n.NEW_LABEL AS LABEL
    FROM AWS_DP_PROD.ML_OPS.LEADIQ_TRAINING_SET_V4 t
    JOIN (SELECT LEAD_ID, MAX(TARGET_CONVERTED) AS NEW_LABEL
          FROM AWS_DP_PROD.ML_OPS.LEADIQ_TARGET_LABELS_V4 GROUP BY LEAD_ID) n
    USING (LEAD_ID)
""").to_pandas()

drop_cols = ['LEAD_ID','LEAD_TS','LABEL','BOAT_LEVEL_POS']
features  = [c for c in df.columns if c not in drop_cols]

df['LEAD_TS'] = pd.to_datetime(df['LEAD_TS'], utc=True).dt.tz_localize(None)
train_end, test_start = pd.Timestamp('2024-04-30'), pd.Timestamp('2024-08-28')
test_end = df['LEAD_TS'].max() - pd.DateOffset(months=15)

tr = df[df['LEAD_TS'] <= train_end]
te = df[(df['LEAD_TS'] >= test_start) & (df['LEAD_TS'] <= test_end)].reset_index(drop=True)
X_tr, y_tr = tr[features], tr['LABEL'].values
X_te, y_te = te[features], te['LABEL'].values
spw = (y_tr==0).sum() / (y_tr==1).sum()
print(f'train {len(tr):,}  test {len(te):,}  test_pos {y_te.mean():.4f}  spw {spw:.0f}')

In [ ]:
from xgboost import XGBClassifier

params = dict(n_estimators=300, max_depth=6, learning_rate=0.1,
              scale_pos_weight=spw, eval_metric='aucpr',
              missing=np.nan, random_state=42, n_jobs=-1)
model = XGBClassifier(**params).fit(X_tr, y_tr)
p_te = model.predict_proba(X_te)[:, 1]

model.save_model('leadiq_persontarget_v1.json')   
print('saved: leadiq_persontarget_v1.json')

In [ ]:
# Kept from the old evaluation cell -- the segment-breakdown cell below still
# calls lift_at_k and roc_auc_score, and the feature-importance cell below
# calls plt.* -- both imports used to live inside the removed evaluation
# cells. Full metrics/plots/decile-lift/threshold analysis now lives in
# evaluation.ipynb, reading from LEADIQ_SCORED_V1 (written at the bottom of
# this notebook), so it isn't duplicated here.
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

def lift_at_k(y, p, k):
    n = max(int(len(y)*k), 1)
    return y[np.argsort(-p)[:n]].mean() / y.mean()

In [ ]:
imp = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print(imp.round(4).to_string())
imp.head(15).iloc[::-1].plot.barh(figsize=(8,6), title='Top 15 feature importance (gain)')
plt.tight_layout(); plt.savefig('importance.png', dpi=150); plt.show()

In [ ]:
src_cols = [c for c in features if c.startswith('SRC_')]
te_seg = te.copy(); te_seg['score'] = p_te
te_seg['pipeline'] = te_seg[src_cols].idxmax(axis=1).str.replace('SRC_','')

rows = []
for seg, g in te_seg.groupby('pipeline'):
    if len(g) < 5000: continue
    r = {'pipeline': seg, 'n': len(g), 'base_rate': g['LABEL'].mean()}
    r['lift@10%_within'] = lift_at_k(g['LABEL'].values, g['score'].values, 0.10)
    r['auc'] = roc_auc_score(g['LABEL'], g['score']) if g['LABEL'].nunique()>1 else np.nan
    rows.append(r)
seg_table = pd.DataFrame(rows).sort_values('n', ascending=False)
print(seg_table.round(3).to_string(index=False))

In [ ]:
import pandas as pd

# Map score -> 1-100 percentile, then -> PRD tier (Hot 75-100 / Warm 40-74 / Cold 1-39)
scored = te[['LEAD_ID', 'LEAD_TS', 'LABEL']].copy()
scored['raw_score'] = p_te
scored['score_1_100'] = (scored['raw_score'].rank(pct=True) * 99 + 1).round(0).astype(int)
scored['tier'] = pd.cut(
    scored['score_1_100'],
    bins=[0, 39, 74, 100],
    labels=['Cold', 'Warm', 'Hot']
)

print(scored['tier'].value_counts())
scored.head(10)

In [ ]:
session.sql("USE ROLE DATA_SCIENTIST").collect()
session.write_pandas(
    scored,
    table_name="LEADIQ_SCORED_V1",
    database="AWS_DP_PROD",
    schema="ML_OPS",
    auto_create_table=True,
    overwrite=True
)